In [1]:
from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras    
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

2024-03-28 08:09:34.207870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-28 08:09:35.338584: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-03-28 08:09:37.187796: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-28 08:09:37.409787: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-28 08:09:37.409839: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-28 08:09:37.579556: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-28 08:09:37.579615: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 13188595093527533503
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 6434692539294320126
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [3]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [4]:
DATA_SIZE = 30000
VALID_DATA_SIZE = DATA_SIZE / 10

ORG_TRAIN_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

ORG_VALID_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

In [6]:
def get_dataset_fromCsv():
    cnt = 0
    
    os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("/root/Data/hangul/dataset/shuffled_tranDataset.csv", 'r', encoding='utf-8')
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        if (cnt > DATA_SIZE): break
        else                : cnt += 1

        yield img, (label1, label2, label3)
        
        
        
def get_valid_dataset_fromCsv():
    cnt = 0

    os.system("shuf /root/Data/hangul/dataset/validation.csv > /root/Data/hangul/dataset/shuffled_validation.csv")    
    csvFile = open("/root/Data/hangul/dataset/shuffled_validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        if (cnt > VALID_DATA_SIZE): break
        else                : cnt += 1

        yield img, (label1, label2, label3)

In [7]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

2024-03-27 14:07:25.584637: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:07:25.584735: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:07:25.584766: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:07:25.585150: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-27 14:07:25.585185: I external/local_xla/xla/stream_executor

In [8]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

l = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(posts_input)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)

l = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(l)
l = layers.BatchNormalization()(l)
l = layers.Activation('relu')(l)
l = layers.MaxPool2D(pool_size=2)(l)


x = layers.Flatten()(l)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [8]:
save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwrite.weights.h5"

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')


#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1711322447.790938    5031 service.cc:145] XLA service 0x7f01a4014c30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1711322447.790991    5031 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-03-25 08:20:47.877832: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-25 08:20:48.223370: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


     24/Unknown 7s 7ms/step - DenseCho2_accuracy: 0.1343 - DenseJong2_accuracy: 0.0369 - DenseJung2_accuracy: 0.1302 - loss: 55.2384       

I0000 00:00:1711322450.917464    5031 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  29993/Unknown 201s 6ms/step - DenseCho2_accuracy: 0.0888 - DenseJong2_accuracy: 0.2125 - DenseJung2_accuracy: 0.0954 - loss: 8.7615

2024-03-25 08:24:05.318101: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-25 08:24:05.318158: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-03-25 08:24:05.318190: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 5735501165122845375
2024-03-25 08:24:05.318204: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8941882810136317589
/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  

KeyboardInterrupt: 

In [9]:
model.fit(dataset, validation_data= validDtaset, epochs = 100, batch_size = 16)

Epoch 1/100


I0000 00:00:1711516054.730623  120003 service.cc:145] XLA service 0x7f646c003540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1711516054.730663  120003 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-03-27 14:07:34.789159: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-27 14:07:35.081474: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


     25/Unknown 6s 7ms/step - DenseCho2_accuracy: 0.0439 - DenseJong2_accuracy: 0.1429 - DenseJung2_accuracy: 0.0000e+00 - loss: 63.0608   

I0000 00:00:1711516057.380033  120003 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  29996/Unknown 198s 6ms/step - DenseCho2_accuracy: 0.0882 - DenseJong2_accuracy: 0.2253 - DenseJung2_accuracy: 0.1041 - loss: 8.7121

...

29995/30001 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - DenseCho2_accuracy: 0.2924 - DenseJong2_accuracy: 0.4947 - DenseJung2_accuracy: 0.3358 - loss: 5.4253

2024-03-27 14:14:05.271844: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-27 14:14:05.271882: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-27 14:14:05.271894: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 7055437652265437426
2024-03-27 14:14:05.271899: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1655848307804283234
2024-03-27 14:14:05.271903: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 6413100108270230508
2024-03-27 14:14:05.271925: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 4430715828632362320


30001/30001 ━━━━━━━━━━━━━━━━━━━━ 196s 7ms/step - DenseCho2_accuracy: 0.2924 - DenseJong2_accuracy: 0.4947 - DenseJung2_accuracy: 0.3358 - loss: 5.4250 - val_DenseCho2_accuracy: 0.5805 - val_DenseJong2_accuracy: 0.5188 - val_DenseJung2_accuracy: 0.5022 - val_loss: 4.2691
Epoch 3/100
   14/30001 ━━━━━━━━━━━━━━━━━━━━ 4:06 8ms/step - DenseCho2_accuracy: 0.7625 - DenseJong2_accuracy: 0.4620 - DenseJung2_accuracy: 0.4926 - loss: 3.3670        

2024-03-27 14:14:13.593425: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-27 14:14:13.593470: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-27 14:14:13.593483: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 7055437652265437426
2024-03-27 14:14:13.593487: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1655848307804283234
2024-03-27 14:14:13.593492: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 6413100108270230508
2024-03-27 14:14:13.593513: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 4430715828632362320


13931/30001 ━━━━━━━━━━━━━━━━━━━━ 1:44 6ms/step - DenseCho2_accuracy: 0.7498 - DenseJong2_accuracy: 0.7060 - DenseJung2_accuracy: 0.6939 - loss: 2.5771